### Atividade de PySpark - 15 exercicios
### Utilizando os arquivos vendas_spark.csv e clientes_spark.csv
### Alunos: Ana e Bruno

In [1]:
# Apache Spark
%pip install -q pyspark

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Em caso de uso do Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

#import os
# os.listdir("/content/drive/MyDrive/Spark")

In [3]:
CAMINHO_VENDAS = r"C:\_Pessoas\Bruno.Araujo\CDN_IFAD2\Atividade Spark\vendas_spark.csv"
CAMINHO_CLIENTES = r"C:\_Pessoas\Bruno.Araujo\CDN_IFAD2\Atividade Spark\clientes_spark.csv"

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Atividade_Spark")  \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark iniciado com sucesso: {spark.version}")

Spark iniciado com sucesso: 4.2.0


## Bloco 1: Leitura e exploracao
#### Exercicio 1: Leitura dos arquivos

In [5]:
df_vendas = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(CAMINHO_VENDAS)

df_clientes = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(CAMINHO_CLIENTES)

In [6]:
df_vendas.show(10)

+--------+----------+----------+--------+-----------+------+----------+--------------+-----------+
|id_venda|id_cliente|data_venda| produto|  categoria|estado|quantidade|preco_unitario|valor_total|
+--------+----------+----------+--------+-----------+------+----------+--------------+-----------+
|       1|      4040|2025-06-27| Monitor|Eletronicos|    MG|         9|        1212.1|    10908.9|
|       2|      3798|2025-03-15| Monitor|Eletronicos|    PR|         1|       1285.72|    1285.72|
|       3|      2513|2026-03-18| Monitor|Eletronicos|    RS|         6|       1242.52|    7455.12|
|       4|      4100|2025-06-24|  Webcam|Eletronicos|    RS|         3|        334.34|    1003.02|
|       5|      4807|2025-02-18| Armario|     Moveis|    SC|         9|       1809.28|   16283.52|
|       6|      2128|2026-04-16|Notebook|Eletronicos|    RJ|         6|       3742.72|   22456.32|
|       7|       359|2026-06-18| Teclado|Eletronicos|    BA|         2|        139.83|     279.66|
|       8|

In [7]:
df_vendas.printSchema()

root
 |-- id_venda: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: double (nullable = true)
 |-- valor_total: double (nullable = true)



In [8]:
df_clientes.show(10)

+----------+--------------------+------+--------------+-------------+
|id_cliente|             cliente|estado|        cidade|data_cadastro|
+----------+--------------------+------+--------------+-------------+
|         1|        Igor Souza 1|    RJ|Rio de Janeiro|   2023-02-21|
|         2|    Daniela Santos 2|    PR|      Curitiba|   2023-10-13|
|         3|        Ana Santos 3|    SC| Florianopolis|   2023-03-07|
|         4|         Ana Souza 4|    PR|      Curitiba|   2024-04-21|
|         5|      Thiago Costa 5|    SC| Florianopolis|   2024-03-27|
|         6|    Renata Pereira 6|    SP|     Sao Paulo|   2023-11-23|
|         7|   Gabriel Pereira 7|    RS|  Porto Alegre|   2023-11-15|
|         8|     Rafael Santos 8|    RJ|Rio de Janeiro|   2023-07-09|
|         9|        Igor Silva 9|    BA|      Salvador|   2024-12-05|
|        10|Daniela Rodrigues 10|    PE|        Recife|   2026-01-03|
+----------+--------------------+------+--------------+-------------+
only showing top 10 

In [9]:
df_clientes.printSchema()

root
 |-- id_cliente: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- data_cadastro: date (nullable = true)



In [10]:
print(f"O df_vendas possui {df_vendas.count()} registros")
print(f"O df_clientes possui {df_clientes.count()} registros")

O df_vendas possui 50000 registros
O df_clientes possui 5000 registros


#### Exercicio 2: Conferindo os tipos

In [11]:
# Verificando o tipo inferido das colunas de data
print(f"data_venda: {df_vendas.schema['data_venda'].dataType}")
print(f"data_cadastro: {df_clientes.schema['data_cadastro'].dataType}")

data_venda: DateType()
data_cadastro: DateType()


In [12]:
from pyspark.sql.functions import col, to_date

# Caso as colunas venham como string, converte para date
if df_vendas.schema["data_venda"].dataType.simpleString() == "string":
    df_vendas = df_vendas.withColumn(
        "data_venda",
        to_date(col("data_venda"), "yyyy-MM-dd")
    )

if df_clientes.schema["data_cadastro"].dataType.simpleString() == "string":
    df_clientes = df_clientes.withColumn(
        "data_cadastro",
        to_date(col("data_cadastro"), "yyyy-MM-dd")
    )

df_vendas.printSchema()
df_clientes.printSchema()

root
 |-- id_venda: integer (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- data_venda: date (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: double (nullable = true)
 |-- valor_total: double (nullable = true)

root
 |-- id_cliente: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- cidade: string (nullable = true)
 |-- data_cadastro: date (nullable = true)



O `inferSchema` ja identificou `data_venda` e `data_cadastro` como `date`, pois estao no formato `yyyy-MM-dd`.
A conversao com `to_date` so e aplicada caso as colunas venham como `string`.

## Bloco 2: Select e filtros (where)
#### Exercicio 3: Selecao de colunas

In [13]:
df_vendas.select(
    "produto",
    "categoria",
    "estado",
    "preco_unitario"
).show(15)

+----------+-----------+------+--------------+
|   produto|  categoria|estado|preco_unitario|
+----------+-----------+------+--------------+
|   Monitor|Eletronicos|    MG|        1212.1|
|   Monitor|Eletronicos|    PR|       1285.72|
|   Monitor|Eletronicos|    RS|       1242.52|
|    Webcam|Eletronicos|    RS|        334.34|
|   Armario|     Moveis|    SC|       1809.28|
|  Notebook|Eletronicos|    RJ|       3742.72|
|   Teclado|Eletronicos|    BA|        139.83|
|   Monitor|Eletronicos|    MG|       1099.74|
|  Notebook|Eletronicos|    RS|       3645.76|
|   Armario|     Moveis|    PE|       2129.52|
|     Mouse|Eletronicos|    PR|         78.16|
|  Notebook|Eletronicos|    SP|       3223.52|
|     Mouse|Eletronicos|    PR|         72.52|
|   Headset|Eletronicos|    PE|        237.04|
|Impressora|Eletronicos|    SC|       1157.62|
+----------+-----------+------+--------------+
only showing top 15 rows


#### Exercicio 4: Filtro simples

In [14]:
vendas_qtde_10 = df_vendas.where(
    df_vendas.quantidade == 10
)

vendas_qtde_10.show()

print(f"Total de vendas com quantidade igual a 10: {vendas_qtde_10.count()}")

+--------+----------+----------+----------+-----------+------+----------+--------------+-----------+
|id_venda|id_cliente|data_venda|   produto|  categoria|estado|quantidade|preco_unitario|valor_total|
+--------+----------+----------+----------+-----------+------+----------+--------------+-----------+
|      17|      1275|2024-02-22|      Mesa|     Moveis|    RS|        10|       1643.33|    16433.3|
|      40|      4409|2025-07-08|   Monitor|Eletronicos|    RS|        10|       1112.87|    11128.7|
|      74|       687|2025-03-15|      Mesa|     Moveis|    RS|        10|       1606.13|    16061.3|
|      96|      4424|2025-02-11|   Monitor|Eletronicos|    BA|        10|        1087.8|    10878.0|
|      98|       891|2025-09-11|    Tablet|Eletronicos|    PR|        10|       1698.83|    16988.3|
|     104|      2916|2025-09-30|   Monitor|Eletronicos|    BA|        10|       1197.42|    11974.2|
|     108|      4629|2024-07-09|   Teclado|Eletronicos|    RJ|        10|        136.39|   

Total de vendas com quantidade igual a 10: 4980


#### Exercicio 5: Filtro com multiplas condicoes

In [15]:
df_vendas.select("id_venda", "produto", "estado", "preco_unitario") \
         .where(
             (df_vendas.preco_unitario >= 200) &
             (df_vendas.preco_unitario < 3000) &
             (df_vendas.estado == "SP")
         ) \
         .show()

+--------+----------+------+--------------+
|id_venda|   produto|estado|preco_unitario|
+--------+----------+------+--------------+
|      29|      Mesa|    SP|       1489.66|
|      59|Smartphone|    SP|       2789.45|
|      62|      Mesa|    SP|       1551.52|
|      65|   Headset|    SP|        246.43|
|      69|   Armario|    SP|       2134.36|
|      72|   Monitor|    SP|       1081.45|
|      95|   Cadeira|    SP|        887.98|
|     114|   Cadeira|    SP|        953.32|
|     129|   Armario|    SP|       2158.92|
|     133|      Mesa|    SP|       1608.02|
|     135|Impressora|    SP|       1055.89|
|     163|   Monitor|    SP|       1284.52|
|     167|    Tablet|    SP|       1876.76|
|     175|    Webcam|    SP|         318.5|
|     191|Impressora|    SP|        993.52|
|     200|    Tablet|    SP|       1825.65|
|     209|Impressora|    SP|       1160.74|
|     211|    Webcam|    SP|        331.28|
|     217|    Webcam|    SP|         331.7|
|     223|   Headset|    SP|    

#### Exercicio 6: Filtro por categoria

In [16]:
# Descobrindo quais categorias existem
df_vendas.select("categoria").distinct().show()

+-----------+
|  categoria|
+-----------+
|Eletronicos|
|     Moveis|
+-----------+



In [17]:
df_vendas.select("produto", "categoria") \
         .where(df_vendas.categoria == "Eletronicos") \
         .distinct() \
         .show()

+----------+-----------+
|   produto|  categoria|
+----------+-----------+
|   Headset|Eletronicos|
|Impressora|Eletronicos|
|Smartphone|Eletronicos|
|   Teclado|Eletronicos|
|  Notebook|Eletronicos|
|    Tablet|Eletronicos|
|   Monitor|Eletronicos|
|     Mouse|Eletronicos|
|    Webcam|Eletronicos|
+----------+-----------+



## Bloco 3: Colunas calculadas
#### Exercicio 7: Validando o valor total

In [18]:
from pyspark.sql.functions import col, abs

df_vendas_validacao = df_vendas.withColumn(
    "valor_calculado",
    col("quantidade") * col("preco_unitario")
)

vendas_divergentes = df_vendas_validacao.where(
    abs(col("valor_calculado") - col("valor_total")) > 0.01
)

vendas_divergentes.select(
    "id_venda",
    "quantidade",
    "preco_unitario",
    "valor_total",
    "valor_calculado"
).show()

print(f"Quantidade de linhas divergentes: {vendas_divergentes.count()}")

+--------+----------+--------------+-----------+---------------+
|id_venda|quantidade|preco_unitario|valor_total|valor_calculado|
+--------+----------+--------------+-----------+---------------+
+--------+----------+--------------+-----------+---------------+

Quantidade de linhas divergentes: 0


**Resultado:** nenhuma linha diverge (0 linhas com diferenca maior que 0,01), ou seja,
a coluna `valor_total` do arquivo e consistente com `quantidade * preco_unitario`.

#### Exercicio 8: Extracao de datas

In [19]:
from pyspark.sql.functions import year, month

df_vendas = df_vendas.withColumn(
    "ano_venda",
    year(col("data_venda"))
).withColumn(
    "mes_venda",
    month(col("data_venda"))
)

df_vendas.select(
    "id_venda",
    "data_venda",
    "ano_venda",
    "mes_venda"
).show(10)

+--------+----------+---------+---------+
|id_venda|data_venda|ano_venda|mes_venda|
+--------+----------+---------+---------+
|       1|2025-06-27|     2025|        6|
|       2|2025-03-15|     2025|        3|
|       3|2026-03-18|     2026|        3|
|       4|2025-06-24|     2025|        6|
|       5|2025-02-18|     2025|        2|
|       6|2026-04-16|     2026|        4|
|       7|2026-06-18|     2026|        6|
|       8|2026-01-13|     2026|        1|
|       9|2024-02-22|     2024|        2|
|      10|2024-09-02|     2024|        9|
+--------+----------+---------+---------+
only showing top 10 rows


## Bloco 4: Agregacoes (groupBy)
#### Exercicio 9: Total por categoria

In [20]:
# GROUP BY

from pyspark.sql.functions import sum

vendas_por_categoria = df_vendas.groupBy(
    "categoria"
).agg(
    sum("valor_total")
    .alias("total_categorizado")
).orderBy(
    col("total_categorizado").desc()
)

vendas_por_categoria.show()

+-----------+--------------------+
|  categoria|  total_categorizado|
+-----------+--------------------+
|Eletronicos|2.5740961157999966E8|
|     Moveis| 1.015882405100003E8|
+-----------+--------------------+



#### Exercicio 10: Resumo estatistico por categoria

In [21]:
from pyspark.sql.functions import(
    sum,
    count,
    avg,
    round
)

df_vendas_valores = df_vendas.groupBy(
    "categoria"
).agg(
    count("id_venda").alias("quantidade"),
    sum("quantidade").alias("qtde_vendida"),
    sum("valor_total").alias("total_vendas"),
    round(avg("valor_total"), 2).alias("media_venda")
)

df_vendas_valores.show()

+-----------+----------+------------+--------------------+-----------+
|  categoria|quantidade|qtde_vendida|        total_vendas|media_venda|
+-----------+----------+------------+--------------------+-----------+
|Eletronicos|     37354|      205564|2.5740961157999966E8|    6891.09|
|     Moveis|     12646|       69381| 1.015882405100003E8|    8033.23|
+-----------+----------+------------+--------------------+-----------+



#### Exercicio 11: Agregacao por duas dimensoes

In [22]:
vendas_estado_categoria = df_vendas.groupBy(
    "estado",
    "categoria"
).agg(
    round(sum("valor_total"), 2).alias("total_vendido")
).orderBy(
    col("total_vendido").desc()
)

vendas_estado_categoria.show(10)

+------+-----------+-------------+
|estado|  categoria|total_vendido|
+------+-----------+-------------+
|    PR|Eletronicos| 3.32631514E7|
|    PE|Eletronicos| 3.27540734E7|
|    RS|Eletronicos|3.248955348E7|
|    MG|Eletronicos|3.202073487E7|
|    RJ|Eletronicos|3.191619186E7|
|    BA|Eletronicos|3.183307703E7|
|    SC|Eletronicos|3.176798799E7|
|    SP|Eletronicos|3.136484155E7|
|    MG|     Moveis|1.331636673E7|
|    RJ|     Moveis|1.316459005E7|
+------+-----------+-------------+
only showing top 10 rows


In [23]:
maior_receita = vendas_estado_categoria.first()

print(f"Maior receita: {maior_receita['estado']} / {maior_receita['categoria']} "
      f"com total de {maior_receita['total_vendido']}")

Maior receita: PR / Eletronicos com total de 33263151.4


**Resultado:** a combinacao estado/categoria com maior receita foi **PR / Eletronicos**.

## Bloco 5: Spark SQL
#### Exercicio 12: Consulta SQL

In [24]:
df_vendas.createOrReplaceTempView("vendas")

In [25]:
resultado = spark.sql("""
    Select produto,
           round(sum(valor_total), 2) as total_vendido,
           count(id_venda) as qtde_vendas
    From vendas
    Group By produto
    Having sum(valor_total) > 1000000
    Order By total_vendido desc
""")

resultado.show()

+----------+-------------+-----------+
|   produto|total_vendido|qtde_vendas|
+----------+-------------+-----------+
|  Notebook|8.030088277E7|       4134|
|Smartphone|6.566517205E7|       4206|
|   Armario|4.561778124E7|       4150|
|    Tablet|4.137258717E7|       4198|
|      Mesa|3.501884558E7|       4178|
|   Monitor|2.701310276E7|       4071|
|Impressora|2.486862679E7|       4139|
|   Cadeira|2.095161369E7|       4318|
|    Webcam|   7310301.48|       4134|
|   Headset|   5655128.09|       4174|
|   Teclado|    3389859.9|       4102|
|     Mouse|   1833950.57|       4196|
+----------+-------------+-----------+



## Bloco 6: Joins
#### Exercicio 13: Inner join

In [26]:
# JOIN = Juncao de tabelas (data frame)

resultado_inner = df_vendas.join(
    df_clientes,
    df_vendas.id_cliente == df_clientes.id_cliente,
    "inner"
)

resultado_select = resultado_inner.select(
    df_vendas["id_venda"],
    df_clientes["cliente"],
    df_clientes["cidade"],
    df_vendas["produto"],
    df_vendas["valor_total"]
)

resultado_select.show()

+--------+--------------------+--------------+----------+-----------+
|id_venda|             cliente|        cidade|   produto|valor_total|
+--------+--------------------+--------------+----------+-----------+
|       1|     Igor Silva 4040|      Curitiba|   Monitor|    10908.9|
|       2|    Bruno Costa 3798|Rio de Janeiro|   Monitor|    1285.72|
|       3|Thiago Rodrigues ...| Florianopolis|   Monitor|    7455.12|
|       4|     Igor Costa 4100|Rio de Janeiro|    Webcam|    1003.02|
|       5|Lucas Rodrigues 4807|      Curitiba|   Armario|   16283.52|
|       6| Bruno Oliveira 2128|  Porto Alegre|  Notebook|   22456.32|
|       7|Helena Rodrigues 359|     Sao Paulo|   Teclado|     279.66|
|       8|  Bruno Pereira 2548|  Porto Alegre|   Monitor|     5498.7|
|       9|Carlos Rodrigues ...|     Sao Paulo|  Notebook|    7291.52|
|      10|    Lucas Silva 4325|  Porto Alegre|   Armario|    2129.52|
|      11| Eduardo Santos 1166|  Porto Alegre|     Mouse|     703.44|
|      12|Thiago Rod

In [27]:
qtde_join = resultado_select.count()
qtde_vendas = df_vendas.count()

print(f"Linhas no inner join: {qtde_join}")
print(f"Linhas no df_vendas: {qtde_vendas}")
print(f"Registros perdidos: {qtde_vendas - qtde_join}")

Linhas no inner join: 50000
Linhas no df_vendas: 50000
Registros perdidos: 0


**Analise:** o inner join retornou 50000 linhas, a mesma quantidade de `df_vendas`, portanto **nao houve perda de registros**.
O inner join mantem apenas as vendas cujo `id_cliente` existe em `df_clientes`; como todas as vendas possuem
um cliente cadastrado, nenhuma linha foi descartada. Caso houvesse vendas com `id_cliente` inexistente (ou nulo)
na tabela de clientes, essas linhas seriam perdidas no inner join.

#### Exercicio 14: Left join e clientes sem compras

In [28]:
resultado_left = df_clientes.join(
    df_vendas,
    df_clientes.id_cliente == df_vendas.id_cliente,
    "left" # semelhante ao left join (outer join) do SQL
)

clientes_sem_compras = resultado_left.where(
    df_vendas["id_venda"].isNull()
).select(
    df_clientes["id_cliente"],
    df_clientes["cliente"],
    df_clientes["data_cadastro"]
)

print(f"Clientes que nunca compraram: {clientes_sem_compras.count()}")

clientes_sem_compras.show(10)

Clientes que nunca compraram: 0
+----------+-------+-------------+
|id_cliente|cliente|data_cadastro|
+----------+-------+-------------+
+----------+-------+-------------+



**Resultado:** nenhum cliente ficou com `id_venda` nulo apos o left join, ou seja, todos os 5000 clientes
cadastrados realizaram ao menos uma compra. Por isso a listagem dos 10 clientes sem compras retorna vazia.

#### Exercicio 15: Join + agregacao (ranking)

In [29]:
top_clientes = df_vendas.join(
    df_clientes,
    df_vendas.id_cliente == df_clientes.id_cliente,
    "inner"
).groupBy(
    df_clientes["id_cliente"],
    df_clientes["cliente"],
    df_clientes["cidade"]
).agg(
    count(df_vendas["id_venda"]).alias("qtde_compras"),
    round(sum(df_vendas["valor_total"]), 2).alias("total_gasto")
).orderBy(
    col("total_gasto").desc()
)

top_clientes.show(10)

+----------+--------------------+--------------+------------+-----------+
|id_cliente|             cliente|        cidade|qtde_compras|total_gasto|
+----------+--------------------+--------------+------------+-----------+
|      1559| Carlos Pereira 1559|Rio de Janeiro|          11|  218835.73|
|       674| Lucas Rodrigues 674|Belo Horizonte|          18|  214957.44|
|      1741|  Daniela Silva 1741|Belo Horizonte|          18|  209374.34|
|      1145| Fernanda Silva 1145|     Sao Paulo|          15|  205682.93|
|      1434|   Renata Souza 1434|Rio de Janeiro|          12|  205615.07|
|      4495|  Carlos Santos 4495|        Recife|          19|  205548.27|
|      1273|Vanessa Almeida 1273|      Salvador|          18|  196255.16|
|      4563|      Ana Silva 4563|     Sao Paulo|          22|  190779.95|
|      2104| Thiago Pereira 2104|Belo Horizonte|          16|  188228.36|
|      2716| Renata Almeida 2716|Rio de Janeiro|          17|  187487.39|
+----------+--------------------+-----